# MusicWave: căn lời bài hát tiếng Việt tự động

Pipeline: **Demucs** (tách giọng) → **ASR** (Whisper, WhisperX, PhoWhisper, wav2vec2) → **căn chỉnh** (ghép quy hoạch động hoặc căn chỉnh cưỡng bức CTC) → **đánh giá** so với mốc ghi tay → **gửi kết quả tốt nhất lên web**.

**Trước khi chạy:** `Runtime → Change runtime type → T4 GPU`.

Mọi kết quả được lưu lên Google Drive sau từng bài. Nếu Colab bị ngắt, chỉ cần chạy lại từ đầu: bài nào đã xong sẽ được bỏ qua.

| Bước | Việc |
|---|---|
| 1 | Cài đặt, nối Drive |
| 2 | Nạp nhạc và lời từ thư mục Drive, tạo file để ghi mốc |
| 3 | *(làm ngoài Colab)* ghi mốc bằng `annotator.html`, tải file JSON lên Drive |
| 4 | Tách giọng Demucs |
| 5 | Chạy Whisper, PhoWhisper, wav2vec2, CTC |
| 6 | Chạy WhisperX (cài riêng) |
| 7 | Đánh giá, vẽ biểu đồ |
| 8 | Gửi kết quả lên web |

## 1. Cài đặt

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, subprocess
from pathlib import Path

WORK = Path('/content/drive/MyDrive/musicwave_align')   # dữ liệu và kết quả
CODE = WORK / 'code'                                     # thư mục research/alignment của repo
WORK.mkdir(parents=True, exist_ok=True)

# Cách 1: repo GitHub công khai thì tự clone. Cách 2: tự tải thư mục research/alignment lên Drive vào CODE.
REPO = 'https://github.com/luthanhdat311021-droid/web-nghe-nhac.git'
if not (CODE / 'musicwave_align').exists():
    r = subprocess.run(['git', 'clone', '--depth=1', REPO, '/content/repo'], capture_output=True, text=True)
    if r.returncode == 0 and Path('/content/repo/research/alignment').exists():
        subprocess.run(['cp', '-r', '/content/repo/research/alignment', str(CODE)], check=True)
    else:
        raise SystemExit(f'Không clone được repo. Hãy tải thư mục research/alignment lên Drive tại: {CODE}')
sys.path.insert(0, str(CODE))
print('Code:', CODE)

In [ ]:
!pip -q install demucs openai-whisper jiwer yt-dlp librosa
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('GPU:', torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'KHÔNG CÓ GPU (đổi runtime sang T4)')

## 2. Nạp dữ liệu

Nhạc và lời đọc từ thư mục **`MyDrive/musicwave_align/data`**: tải nguyên thư mục `research/alignment/data` (gồm `audio/`, `lyrics/`, `manifest.json`) lên Drive vào đó. Dữ liệu chỉ dùng nội bộ cho nghiên cứu, **không đưa lên web và không chia sẻ công khai**.

Mỗi bài gồm `audio/<tên>.mp3` và `lyrics/<tên>.txt` (lời, mỗi câu một dòng). Thêm bài mới: chép hai file đó vào rồi thêm một mục vào `manifest.json`.

`DEV_IDS` (khoảng 1/4 số bài) dùng để thử và chỉnh tham số; các bài còn lại là tập kiểm tra, **không chỉnh tham số theo tập này**.

In [ ]:
import json, shutil
from musicwave_align.pipeline import song_lines

DATA = WORK / 'data'
# file lớn có thể được tải lên thành nhiều phần "<tên>.mp3.part1", ".part2"...: ghép lại
for first in (DATA / 'audio').glob('*.part1'):
    target = first.with_suffix('')
    if not target.exists():
        parts = sorted(first.parent.glob(target.name + '.part*'), key=lambda p: int(p.suffix[5:]))
        target.write_bytes(b''.join(p.read_bytes() for p in parts))
        print('Đã ghép', target.name, 'từ', len(parts), 'phần')
manifest =json.loads((DATA / 'manifest.json').read_text(encoding='utf-8'))
songs, mix = [], {}
for m in manifest:
    lyr = DATA / m['lyrics']
    if not lyr.exists() or not (DATA / m['audio']).exists():
        print('BỎ (thiếu nhạc hoặc lời):', m['title']); continue
    s = {'id': m['id'], 'title': m['title'], 'artist': {'name': m.get('artist')},
         'genre': {'name': m.get('genre')}, 'lyrics': {'plainLyrics': lyr.read_text(encoding='utf-8')}}
    if len(song_lines(s)) < 4:
        print('BỎ (lời quá ngắn):', m['title']); continue
    songs.append(s)
    mix[s['id']] = DATA / m['audio']
DEV_IDS = [s['id'] for s in songs[::4]]
print(len(songs), 'bài,', len(DEV_IDS), 'bài dùng để chỉnh tham số')

In [ ]:
ANNOT = WORK / 'annotate'          # tải thư mục này về máy để ghi mốc bằng annotator.html
ANNOT.mkdir(exist_ok=True)
for s in songs:
    (ANNOT / f"{s['id']}.json").write_text(json.dumps(
        {'songId': s['id'], 'title': s['title'], 'lines': song_lines(s)}, ensure_ascii=False, indent=1), encoding='utf-8')
print('File lời để ghi mốc nằm ở Drive: musicwave_align/annotate (nhạc dùng luôn file trong data/audio)')

## 3. Ghi mốc chuẩn (làm ngoài Colab)

1. Tải thư mục `musicwave_align/annotate` từ Drive về máy.
2. Mở `research/alignment/tools/annotator.html` bằng trình duyệt.
3. Với mỗi bài: mở file nhạc và file `.json` cùng tên, phát nhạc, bấm **Space** đúng lúc ca sĩ bắt đầu mỗi dòng.
   Chọn 10 bài ghi thêm **theo từ** để đánh giá karaoke.
4. Bấm **Xuất JSON**, tải các file xuất ra lên Drive vào `musicwave_align/ground_truth/`.

Các bước 4–6 không cần chờ ghi mốc xong; chạy song song được.

## 4. Tách giọng bằng Demucs (khoảng 30–60 giây mỗi bài)

In [ ]:
from musicwave_align.web import separate_vocals
vocals = {}
for sid, path in mix.items():
    try:
        vocals[sid] = separate_vocals(path, WORK / 'demucs')
    except Exception as e:
        print('Demucs lỗi', sid, e)
INPUTS = {'mix': mix, 'vocals': vocals}
print(len(vocals), 'bài đã tách giọng')

## 5. Chạy các mô hình

Mỗi mô hình chạy trên hai đầu vào: bản gốc (`mix`) và giọng đã tách (`vocals`). Chạy xong một mô hình thì giải phóng GPU rồi mới nạp mô hình sau.

In [ ]:
from pathlib import Path
from musicwave_align.asr import Whisper, PhoWhisper, Wav2Vec2
from musicwave_align.pipeline import run_asr, run_ctc

m = Whisper(device=DEVICE)
run_asr(m, songs, INPUTS, WORK); m.close()

In [ ]:
m = PhoWhisper(device=DEVICE)
run_asr(m, songs, INPUTS, WORK); m.close()

In [ ]:
w2v = Wav2Vec2(device=DEVICE)
run_asr(w2v, songs, INPUTS, WORK)     # wav2vec2 như một mô hình ASR + ghép quy hoạch động
run_ctc(w2v, songs, INPUTS, WORK)     # căn chỉnh cưỡng bức CTC
w2v.close()

## 6. WhisperX

WhisperX có thể cài phiên bản PyTorch khác. Sau khi cài: **Runtime → Restart session**, rồi chạy lại bước 1, ô đầu của bước 2 (`mw = ...`), ô chọn bài, ô tải nhạc, bước 4 (bài đã xử lý sẽ được bỏ qua), sau đó chạy ô thứ hai dưới đây.

In [ ]:
!pip -q install whisperx

In [ ]:
from musicwave_align.asr import WhisperX
from musicwave_align.pipeline import run_asr
m = WhisperX(device=DEVICE)
run_asr(m, songs, INPUTS, WORK); m.close()

## 7. Đánh giá

- **dòng / từ mean_s**: độ lệch trung bình (giây) so với mốc ghi tay.
- **within_0.3s_%**: tỷ lệ dòng (từ) lệch không quá 0,3 giây.
- **wer / cer**: tỷ lệ lỗi từ / ký tự của mô hình ASR so với lời chuẩn (không áp dụng cho CTC và web hiện tại).
- **seconds**: thời gian xử lý trung bình mỗi bài trên T4.

In [ ]:
from musicwave_align.metrics import load_ground_truth
from musicwave_align.pipeline import evaluate

gt = load_ground_truth(WORK / 'ground_truth')
print(len(gt), 'bài đã có mốc chuẩn')
test_gt = {k: v for k, v in gt.items() if k not in DEV_IDS}

table, per_song = evaluate(songs, test_gt, WORK)
table.to_csv(WORK / 'ket_qua_tap_kiem_tra.csv')
per_song.to_csv(WORK / 'ket_qua_tung_bai.csv', index=False)
table.style.format(precision=2)

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
table['dòng mean_s'].sort_values().plot.barh(ax=axes[0], color='#2563eb')
axes[0].set_title('Độ lệch trung bình đầu dòng (giây, thấp hơn là tốt hơn)')
table['dòng within_0.3s_%'].sort_values().plot.barh(ax=axes[1], color='#16a34a')
axes[1].set_title('% dòng lệch ≤ 0,3 giây (cao hơn là tốt hơn)')
plt.tight_layout(); plt.savefig(WORK / 'bieu_do_so_sanh.png', dpi=150); plt.show()

In [ ]:
# Theo dòng nhạc: cấu hình nào hợp với rap, ballad, ...
per_song.pivot_table(index='genre', columns='config', values='line_mean_s', aggfunc='mean').round(2)

## 8. Gửi kết quả lên web

Cần **token của tài khoản admin**: đăng nhập web bằng tài khoản admin, mở DevTools (F12) → Console, gõ `localStorage.getItem('musicwave_token')` và sao chép chuỗi (không có dấu nháy).

Token chỉ được nhập vào ô ẩn bên dưới, không lưu trong notebook. **Không chia sẻ token cho ai.**

Server phải đã được deploy bản có API `PUT /api/admin/songs/:id/aligned-lyrics`.

In [ ]:
BEST = table.index[0]          # cấu hình tốt nhất trên tập kiểm tra; có thể gán tay, ví dụ 'ctc_vocals'
print('Cấu hình sẽ gửi lên web:', BEST)

In [ ]:
from getpass import getpass
from musicwave_align.web import MusicWave
admin = MusicWave(API, token=getpass('Token admin: '))
import re
for s in songs:
    f = WORK / 'align' / BEST / f"{s['id']}.json"
    # chỉ gửi bài đã có trên web (id dạng UUID của web); bài nghiên cứu nội bộ không đưa lên web
    if not f.exists() or not re.fullmatch(r'[0-9a-f-]{36}', s['id']):
        continue
    lines = json.loads(f.read_text(encoding='utf-8'))['lines']
    if not lines:
        print('Bỏ (không căn được):', s['title']); continue
    try:
        admin.upload_aligned(s['id'], lines)
        print('Đã gửi:', s['title'])
    except Exception as e:
        print('Lỗi', s['title'], e)